## Príprava event datasetu

Súbor: priprava_event_dat.ipynb

Program: Hospodárska informatika

Vypracovala: Bc. Terézia Drengubiaková

Bakalárska práca: : Metódy strojového učenia pre včasnú predikciu geomagnetických búrok

Vedúci diplomovej práce: doc. Ing. Peter Butka, PhD.

Konzultanti: Ing. Viera Krešňáková, PhD., RNDr. Šimon Mackovjak, PhD.

In [1]:
import pandas as pd

In [10]:
def ensure_datetime_index(df, time_col="time1"):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        if time_col not in df.columns:
            raise ValueError(f"df nemá DatetimeIndex ani stĺpec '{time_col}'")
        df[time_col] = pd.to_datetime(df[time_col], utc=True)
        df = df.set_index(time_col)
    else:
        # nech je to UTC-aware, ak je naivné
        if df.index.tz is None:
            df.index = df.index.tz_localize("UTC")
        else:
            df.index = df.index.tz_convert("UTC")
    df = df.sort_index()
    return df

In [11]:
def parse_utc(s):
    # formát z tabuľky: "YYYY-MM-DD HH:MM" (v PDF sú pomlčky/typografia, tu dávame normálne)
    return pd.Timestamp(s, tz="UTC")

In [12]:
def fix_end_before_start(start, end):
    # V Table 5 je pri niektorých riadkoch zjavne "odrezaný deň" (end < start).
    # Ak sa to stane, posunieme end o +1 deň.
    if end <= start:
        end = end + pd.Timedelta(days=1)
    return end

EVENTY

In [13]:
EVENTS = [
    (1,  "1998-05-04 03:00", "1998-05-04 08:00"),
    (2,  "1998-08-26 22:00", "1998-08-27 08:00"),
    (3,  "1998-09-25 00:00", "1998-09-25 09:00"),
    (4,  "1998-08-11 03:00", "1998-08-11 05:00"),
    (5,  "1999-09-22 21:00", "1999-09-22 23:00"),
    (6,  "1999-10-22 03:00", "1999-10-22 08:00"),
    (7,  "2000-04-06 18:00", "2000-04-07 04:00"),
    (8,  "2000-05-24 00:00", "2000-05-24 05:00"),
    (9,  "2000-07-15 12:00", "2000-07-16 05:00"),
    (10, "2000-08-12 03:00", "2000-08-12 11:00"),
    (11, "2000-09-17 21:00", "2000-09-18 02:00"),
    (12, "2001-03-31 03:00", "2001-04-01 02:00"),
    (13, "2001-04-11 15:00", "2001-04-12 02:00"),
    (14, "2001-11-05 23:00", "2001-11-06 16:00"),
    (15, "2001-11-24 06:00", "2001-11-24 08:00"),
    (16, "2001-11-24 12:00", "2001-11-24 18:00"),
    (17, "2002-05-23 11:00", "2002-05-23 17:00"),
    (18, "2003-05-29 15:00", "2003-05-29 23:00"),
    (19, "2003-10-29 06:00", "2003-10-30 07:00"),
    (20, "2003-10-30 17:00", "2003-10-31 05:00"),
    (21, "2003-11-20 12:00", "2003-11-21 02:00"),
    (22, "2004-07-27 00:00", "2004-07-27 17:00"),
    (23, "2004-11-07 21:00", "2004-11-08 11:00"),
    (24, "2004-11-09 18:00", "2004-11-10 13:00"),
    (25, "2005-01-07 21:00", "2005-01-07 23:00"),
    (26, "2005-01-18 01:00", "2005-01-18 08:00"),
    (27, "2005-01-21 17:00", "2005-01-21 20:00"),
    (28, "2005-05-15 06:00", "2005-05-15 11:00"),
    (29, "2005-08-24 07:00", "2005-08-24 11:00"),
    (30, "2005-09-11 06:00", "2005-09-11 08:00"),
    (31, "2006-12-14 21:00", "2006-12-15 05:00"),
    (32, "2011-08-05 21:00", "2011-08-05 23:00"),
    (33, "2012-03-09 03:00", "2012-03-09 12:00"),
    (34, "2013-10-02 02:00", "2013-10-02 05:00"),
    (35, "2015-03-17 21:00", "2015-03-17 23:00"),
    (36, "2015-06-22 18:00", "2015-06-22 20:00"),
    (37, "2017-09-07 21:00", "2017-09-08 02:00"),
    (38, "2023-04-24 04:00", "2023-04-24 06:00"),
]

Vyrob event windows

In [14]:
def build_meswe_like_windows(df, dst_col="dst", days=10):
    df = ensure_datetime_index(df)
    if dst_col not in df.columns:
        raise ValueError(f"Chýba stĺpec '{dst_col}' v df")

    windows = []
    for no, s, e in EVENTS:
        start = parse_utc(s)
        end = fix_end_before_start(start, parse_utc(e))

        seg = df.loc[start:end]
        if seg.empty:
            windows.append({
                "event_no": no,
                "start": start, "end": end,
                "tmin": pd.NaT,
                "win_start": pd.NaT, "win_end": pd.NaT,
                "n_rows": 0,
                "note": "V danom intervale nie sú dáta"
            })
            continue

        # lokálne minimum Dst v storm intervale
        tmin = seg[dst_col].idxmin()

        win_start = tmin - pd.Timedelta(days=days)
        win_end   = tmin + pd.Timedelta(days=days)

        # finálny výrez
        w = df.loc[win_start:win_end]
        windows.append({
            "event_no": no,
            "start": start, "end": end,
            "tmin": tmin,
            "win_start": win_start, "win_end": win_end,
            "n_rows": int(len(w)),
            "note": ""
        })

    return pd.DataFrame(windows)


Zostrihni df na event-based dataset

In [15]:
def slice_union_of_windows(df, windows_df):
    df = ensure_datetime_index(df)
    parts = []
    for _, r in windows_df.dropna(subset=["win_start","win_end"]).iterrows():
        parts.append(df.loc[r["win_start"]:r["win_end"]].assign(event_no=int(r["event_no"])))
    if not parts:
        return df.iloc[0:0].copy()
    out = pd.concat(parts).sort_index()
    return out

In [16]:
df = pd.read_csv('DP_omni.csv')

In [18]:
windows_df = build_meswe_like_windows(df, dst_col="dst", days=10)

In [19]:
df_events = slice_union_of_windows(df, windows_df)

In [26]:
df_events

,bz_gsm,v,dst,event_no
time1,,,,
1998-04-24 05:30:00+00:00,-6.7,444.0,-56,1
1998-04-24 06:30:00+00:00,-6.9,442.0,-59,1
1998-04-24 07:30:00+00:00,-4.4,435.0,-69,1
1998-04-24 08:30:00+00:00,-1.5,437.0,-50,1
1998-04-24 09:30:00+00:00,-4.0,452.0,-38,1
...,...,...,...,...
2023-05-04 01:30:00+00:00,-1.9,433.0,-8,38
2023-05-04 02:30:00+00:00,-1.9,437.0,-9,38
2023-05-04 03:30:00+00:00,-1.9,428.0,-7,38


In [25]:
df_events = df_events.drop(columns = ['Unnamed: 0'])

In [27]:
df_events.to_csv("events_omni.csv")